# 🔷 Delta Lake com Apache Spark

## Cenário: E-commerce de Produtos Eletrônicos

### Modelo ER
```
┌──────────────────┐       ┌──────────────────┐
│    clientes      │       │     pedidos      │
│──────────────────│       │──────────────────│
│ id_cliente (PK)  │──────<│ id_pedido (PK)   │
│ nome             │       │ id_cliente (FK)  │
│ email            │       │ produto          │
│ cidade           │       │ quantidade       │
└──────────────────┘       │ valor_total      │
                           │ status           │
                           │ data_pedido      │
                           └──────────────────┘
```

### DDL (referência conceitual)
```sql
CREATE TABLE clientes (
    id_cliente INT,
    nome       STRING,
    email      STRING,
    cidade     STRING
);

CREATE TABLE pedidos (
    id_pedido   INT,
    id_cliente  INT,
    produto     STRING,
    quantidade  INT,
    valor_total DOUBLE,
    status      STRING,
    data_pedido DATE
);
```

## 1. Configuração do Ambiente

In [1]:
import pyspark
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from datetime import date

# Configurar SparkSession com Delta Lake
builder = (
    SparkSession.builder
    .appName('Delta Lake - E-commerce')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print(f'PySpark versão: {pyspark.__version__}')
print('✅ SparkSession com Delta Lake iniciada com sucesso!')

:: loading settings :: url = jar:file:/home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabrielmaciel/.ivy2/cache
The jars for the packages stored in: /home/gabrielmaciel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f88e5945-9f20-4032-b667-32c1f67a775e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (2035ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (87ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (172ms)
:: resolution report :: res

PySpark versão: 3.5.1
✅ SparkSession com Delta Lake iniciada com sucesso!


## 2. INSERT — Inserindo dados iniciais

In [2]:
# Dados de clientes
dados_clientes = [
    (1, 'Ana Souza',    'ana@email.com',    'São Paulo'),
    (2, 'Bruno Lima',   'bruno@email.com',  'Rio de Janeiro'),
    (3, 'Carla Melo',   'carla@email.com',  'Curitiba'),
    (4, 'Diego Faria',  'diego@email.com',  'Belo Horizonte'),
]

df_clientes = spark.createDataFrame(
    dados_clientes,
    ['id_cliente', 'nome', 'email', 'cidade']
)

# Escrever como tabela Delta
df_clientes.write.format('delta').mode('overwrite').save('/tmp/delta/clientes')
print('✅ Tabela clientes criada com Delta Lake')
df_clientes.show()

✅ Tabela clientes criada com Delta Lake
+----------+-----------+---------------+--------------+
|id_cliente|       nome|          email|        cidade|
+----------+-----------+---------------+--------------+
|         1|  Ana Souza|  ana@email.com|     São Paulo|
|         2| Bruno Lima|bruno@email.com|Rio de Janeiro|
|         3| Carla Melo|carla@email.com|      Curitiba|
|         4|Diego Faria|diego@email.com|Belo Horizonte|
+----------+-----------+---------------+--------------+



In [3]:
# Dados de pedidos
dados_pedidos = [
    (101, 1, 'Notebook Dell',   1, 4500.00, 'aprovado',  '2024-01-10'),
    (102, 2, 'iPhone 15',       1, 5800.00, 'aprovado',  '2024-01-11'),
    (103, 3, 'Smart TV 55"',    1, 2900.00, 'pendente',  '2024-01-12'),
    (104, 4, 'Fone Bluetooth',  2,  350.00, 'aprovado',  '2024-01-13'),
    (105, 1, 'SSD 1TB',         1,  480.00, 'pendente',  '2024-01-14'),
]

df_pedidos = spark.createDataFrame(
    dados_pedidos,
    ['id_pedido', 'id_cliente', 'produto', 'quantidade', 'valor_total', 'status', 'data_pedido']
)

df_pedidos.write.format('delta').mode('overwrite').save('/tmp/delta/pedidos')
print('✅ Tabela pedidos criada com Delta Lake')
df_pedidos.show()

✅ Tabela pedidos criada com Delta Lake
+---------+----------+--------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|       produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+--------------+----------+-----------+--------+-----------+
|      101|         1| Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      102|         2|     iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      103|         3|  Smart TV 55"|         1|     2900.0|pendente| 2024-01-12|
|      104|         4|Fone Bluetooth|         2|      350.0|aprovado| 2024-01-13|
|      105|         1|       SSD 1TB|         1|      480.0|pendente| 2024-01-14|
+---------+----------+--------------+----------+-----------+--------+-----------+



## 3. UPDATE — Atualizando registros com DeltaTable

In [4]:
from delta.tables import DeltaTable

# Carregar a tabela Delta
delta_pedidos = DeltaTable.forPath(spark, '/tmp/delta/pedidos')

# UPDATE: aprovar todos os pedidos 'pendente'
delta_pedidos.update(
    condition = col('status') == 'pendente',
    set       = {'status': lit('aprovado')}
)

print('✅ UPDATE executado — todos os pedidos pendentes foram aprovados')
spark.read.format('delta').load('/tmp/delta/pedidos').show()

✅ UPDATE executado — todos os pedidos pendentes foram aprovados


+---------+----------+--------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|       produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+--------------+----------+-----------+--------+-----------+
|      104|         4|Fone Bluetooth|         2|      350.0|aprovado| 2024-01-13|
|      101|         1| Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      103|         3|  Smart TV 55"|         1|     2900.0|aprovado| 2024-01-12|
|      102|         2|     iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      105|         1|       SSD 1TB|         1|      480.0|aprovado| 2024-01-14|
+---------+----------+--------------+----------+-----------+--------+-----------+



## 4. DELETE — Removendo registros

In [5]:
# DELETE: remover pedido de baixo valor (valor_total < 400)
delta_pedidos.delete(condition = col('valor_total') < 400)

print('✅ DELETE executado — pedidos com valor < R$400 removidos')
spark.read.format('delta').load('/tmp/delta/pedidos').show()

✅ DELETE executado — pedidos com valor < R$400 removidos


+---------+----------+-------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|      produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+-------------+----------+-----------+--------+-----------+
|      101|         1|Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      103|         3| Smart TV 55"|         1|     2900.0|aprovado| 2024-01-12|
|      102|         2|    iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      105|         1|      SSD 1TB|         1|      480.0|aprovado| 2024-01-14|
+---------+----------+-------------+----------+-----------+--------+-----------+



## 5. MERGE (UPSERT) — Insert ou Update em uma operação

In [6]:
# Novos pedidos para fazer MERGE
novos_pedidos = [
    (102, 2, 'iPhone 15',    1, 5900.00, 'cancelado', '2024-01-11'),  # UPDATE
    (106, 3, 'Tablet iPad',  1, 3200.00, 'aprovado',  '2024-01-15'),  # INSERT
]

df_novos = spark.createDataFrame(
    novos_pedidos,
    ['id_pedido', 'id_cliente', 'produto', 'quantidade', 'valor_total', 'status', 'data_pedido']
)

delta_pedidos.alias('destino').merge(
    df_novos.alias('origem'),
    'destino.id_pedido = origem.id_pedido'
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print('✅ MERGE executado com sucesso!')
spark.read.format('delta').load('/tmp/delta/pedidos').orderBy('id_pedido').show()

✅ MERGE executado com sucesso!


+---------+----------+-------------+----------+-----------+---------+-----------+
|id_pedido|id_cliente|      produto|quantidade|valor_total|   status|data_pedido|
+---------+----------+-------------+----------+-----------+---------+-----------+
|      101|         1|Notebook Dell|         1|     4500.0| aprovado| 2024-01-10|
|      102|         2|    iPhone 15|         1|     5900.0|cancelado| 2024-01-11|
|      103|         3| Smart TV 55"|         1|     2900.0| aprovado| 2024-01-12|
|      105|         1|      SSD 1TB|         1|      480.0| aprovado| 2024-01-14|
|      106|         3|  Tablet iPad|         1|     3200.0| aprovado| 2024-01-15|
+---------+----------+-------------+----------+-----------+---------+-----------+



## 6. Time Travel — Viagem no tempo com Delta Lake

In [7]:
# Ver histórico de versões da tabela
delta_pedidos.history().select('version', 'timestamp', 'operation').show(truncate=False)

# Ler a versão 0 (estado inicial)
print('\n📜 Estado inicial da tabela (versão 0):')
spark.read.format('delta').option('versionAsOf', 0).load('/tmp/delta/pedidos').show()

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|3      |2026-04-27 16:54:43.506|MERGE    |
|2      |2026-04-27 16:54:30.37 |DELETE   |
|1      |2026-04-27 16:54:19.747|UPDATE   |
|0      |2026-04-27 16:53:59.869|WRITE    |
+-------+-----------------------+---------+


📜 Estado inicial da tabela (versão 0):


+---------+----------+--------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|       produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+--------------+----------+-----------+--------+-----------+
|      104|         4|Fone Bluetooth|         2|      350.0|aprovado| 2024-01-13|
|      101|         1| Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      103|         3|  Smart TV 55"|         1|     2900.0|pendente| 2024-01-12|
|      102|         2|     iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      105|         1|       SSD 1TB|         1|      480.0|pendente| 2024-01-14|
+---------+----------+--------------+----------+-----------+--------+-----------+



## 7. Verificando arquivos Delta no storage

In [8]:
import os

print('📁 Estrutura de arquivos Delta Lake:')
for root, dirs, files in os.walk('/tmp/delta/pedidos'):
    level = root.replace('/tmp/delta/pedidos', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

📁 Estrutura de arquivos Delta Lake:
pedidos/
  part-00002-dcfa0c97-c084-49c4-bce6-92786900d956-c000.snappy.parquet
  .part-00009-b7c6a45c-a6c5-4454-a7f1-fc535d7a1d68-c000.snappy.parquet.crc
  part-00000-8ab1fd98-375c-42fa-b283-cda355a1fe2d-c000.snappy.parquet
  part-00000-52edf41f-aa29-4d20-b928-f703e2652f49-c000.snappy.parquet
  .part-00007-6b1966ff-67a7-47f4-9f26-04c50eaa0eb4-c000.snappy.parquet.crc
  .part-00001-7f1708cc-505b-4d99-b9da-6f7b11adb231-c000.snappy.parquet.crc
  .part-00011-1b6b804f-fc46-4f07-9d80-63630b7848b5-c000.snappy.parquet.crc
  .part-00000-8ab1fd98-375c-42fa-b283-cda355a1fe2d-c000.snappy.parquet.crc
  .part-00000-52edf41f-aa29-4d20-b928-f703e2652f49-c000.snappy.parquet.crc
  .part-00000-3a56d0a9-b985-43dd-98f0-cd133d2e7e51-c000.snappy.parquet.crc
  part-00000-b7cb7c1d-60c2-40fe-a4bd-84515551658c-c000.snappy.parquet
  part-00004-627fc023-615b-4d3f-b14a-d3adb30202ad-c000.snappy.parquet
  .part-00002-dcfa0c97-c084-49c4-bce6-92786900d956-c000.snappy.parquet.crc
  par